<a href="https://colab.research.google.com/github/Przemo75/git_training/blob/master/Haki_update_OK_10_OK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import re
import math

DEBUG = False
def dbg(*args):
    if DEBUG:
        print(*args)

# 🔥 KLUCZOWA OPCJA → NIE USUWAJ SPACJI PO PRZECINKU
base = pd.read_csv('HAKI_new.csv', dtype=str, skipinitialspace=False)
new  = pd.read_csv('oferta_new.csv', dtype=str, skipinitialspace=False)

base.columns = [c.strip() for c in base.columns]
new.columns = [c.strip() for c in new.columns]


# ---------------------------------------------------------
# 🔥 FUNKCJE POMOCNICZE
# ---------------------------------------------------------

def remove_units(val):
    val = str(val)
    # USUWAMY TYLKO kg / kN / KkN — BEZ "h"
    val = re.sub(r'(\d+)\s*(kg|kN|KkN)', r'\1', val, flags=re.IGNORECASE)
    val = re.sub(r'\b(kg|kN|KkN)\b', '', val, flags=re.IGNORECASE)
    return val


def clean_text(val):
    if pd.isna(val) or val is None:
        return ''
    val = str(val)
    val = remove_units(val)
    val = val.replace('"', '')
    val = re.sub(r'\.\s+,', '.,', val)
    val = re.sub(r'\bdostawczy\s*VAN\s*,', 'dostawczy, VAN', val, flags=re.IGNORECASE)
    val = re.sub(r'\s+', ' ', val).strip()
    if val.upper() == 'XX':
        return ''
    return val


# ---------------------------------------------------------
# 🔥 MODEL — POPRAWIONA WERSJA
# ---------------------------------------------------------

def clean_model(val):
    if pd.isna(val) or val is None:
        return ''
    val = str(val)
    val = val.replace('"', '')

    val = remove_units(val)

    # 🔥 SPECJALNY FIX: "R I O" → "RIO"
    val = re.sub(r'\bR\s+I\s+O\b', 'RIO', val, flags=re.IGNORECASE)

    # usuń problematyczne słowa — BEZ "VER"
    val = re.sub(
        r'\b(AUT\.?|AUTOMAT|AUTOMAT\s+VERTICAL|AUT\.?\s*VERTICAL|AUTTOMAT\s+VERTICAL|AUT\.VERTICAL|VERT\.?|VERTICAL|VARTICAL|HORIZONTAL|VERTIC|VERTICA)\b',
        '',
        val, flags=re.IGNORECASE
    )

    # 🔥 SPECJALNE ZASADY TYLKO DLA MODELI ZE STAREX
    u = val.upper()
    if 'STAREX' in u:
        # 1 / H200 / H1 STAREX → H-1 / H200 / H1 STAREX
        val = re.sub(r'^\s*1\s*/', 'H-1 /', val)

        # H-1 / 300 / H1 STAREX → H-1 / H300 / H1 STAREX
        val = re.sub(r'(/)\s*(\d{3})\b', r'\1 H\2', val)

        # 200 / SATELITE / STAREX → H200 / SATELITE / STAREX
        val = re.sub(r'^\s*(\d{3})\b', r'H\1', val)

    # spacje przy myślnikach (S - MAX → S-MAX, H - 1 → H-1)
    val = re.sub(r'\s*-\s*', '-', val)

    # reszta reguł
    val = re.sub(r'\.(?!,)', ' ', val)
    val = re.sub(r'\bE\s*TRON\b', 'E-TRON', val, flags=re.IGNORECASE)
    val = re.sub(r'\bETRON\b', 'E-TRON', val, flags=re.IGNORECASE)
    val = re.sub(r'\bE\s*-\s*TRON\b', 'E-TRON', val, flags=re.IGNORECASE)
    val = re.sub(r'\bCCROSSER\b', 'C-CROSSER', val, flags=re.IGNORECASE)
    val = re.sub(r'\b([A-Z])\s+(\d+)\b', r'\1\2', val)
    val = re.sub(r'\bi\s+(\d+)', r'i\1', val, flags=re.IGNORECASE)
    val = re.sub(r'ID\s*\.\s*(\d+)', r'ID\1', val, flags=re.IGNORECASE)
    val = re.sub(r'\s*,\s*', ' / ', val)
    val = re.sub(r'\s*/\s*', ' / ', val)
    val = re.sub(r'\b3\s*/\s*5\s*drz\.?', '3/5 drz.', val, flags=re.IGNORECASE)
    val = re.sub(r'\b3\s*drz\.?', '3 drz.', val, flags=re.IGNORECASE)
    val = re.sub(r'\b4\s*drz\.?', '4 drz.', val, flags=re.IGNORECASE)
    val = re.sub(r'\b5\s*drz\.?', '5 drz.', val, flags=re.IGNORECASE)
    val = re.sub(r'drz\.\s*/', 'drz., ', val, flags=re.IGNORECASE)
    val = re.sub(r'\b([A-Z])\s*-\s*KLASA\b', r'\1-KLASA', val, flags=re.IGNORECASE)

    val = re.sub(r'\s+', ' ', val).strip()
    return val


# ---------------------------------------------------------
# 🔥 DODATKOWA POPRAWKA TYLKO DLA HYUNDAI (H100/H200/H300/H350)
# ---------------------------------------------------------

def fix_hyundai_model(val):
    if pd.isna(val) or val is None:
        return ''
    s = str(val).strip()
    up = s.upper()

    if re.fullmatch(r'\d{3}', up) and up in ['100', '200', '300', '350']:
        return 'H' + up

    return val


# ---------------------------------------------------------
# 🔥 CZYSZCZENIE KOLUMN NUMERYCZNYCH
# ---------------------------------------------------------

def clean_numeric(val):
    if pd.isna(val) or val is None:
        return ''
    s = str(val)

    s = re.sub(r'[A-Za-z]', '', s)   # usuń litery
    s = s.replace(' ', '')           # usuń spacje
    s = s.replace('.', ',')          # kropka → przecinek

    m = re.findall(r'\d+,?\d*', s)
    if not m:
        return ''
    return m[0]


# ---------------------------------------------------------
# 🔥 ROK PRODUKCJI — poprawka litera+cyfra
# ---------------------------------------------------------

def format_rok_produkcji(val):
    if pd.isna(val) or val is None:
        return ''
    s = str(val)

    # litera + cyfra → litera spacja cyfra
    s = re.sub(r'([A-Za-z])(\d)', r'\1 \2', s)

    s = re.sub(r'\s*/\s*', ' / ', s)
    s = re.sub(r'\s*\.\s*', '.', s)
    return s.strip()


# ---------------------------------------------------------
# 🔥 HOMOLOGACJA, NR KAT., WIĄZKI
# ---------------------------------------------------------

def clean_price(val):
    if pd.isna(val) or val is None or str(val).strip() == '':
        return '0'
    val = str(val).replace(',', '.').replace(' ', '').replace('zł', '')
    try:
        f = float(re.findall(r'\d+\.?\d*', val)[0])
        return str(int(math.ceil(f)))
    except:
        return '0'


def format_time(val):
    if pd.isna(val) or val is None or str(val).strip() == '':
        return '0,0'
    try:
        s = str(val).replace(' ', '').replace(',', '.')
        num = float(re.findall(r'\d+\.?\d*', s)[0])
        return f"{num:.1f}".replace('.', ',')
    except:
        return '0,0'


def format_homologation(val):
    if not val:
        return ''
    raw = str(val)
    if not raw.startswith('E'):
        return raw
    s = raw.upper()
    s = re.sub(r'[^A-Z0-9]', '', s)
    if len(s) < 12:
        return raw
    try:
        e = s[0]
        country = s[1:3]
        rcode = s[3:6]
        zero_x = s[6:8]
        last4 = s[8:12]
        return f"{e}{country}-{rcode}-{zero_x} {last4}"
    except:
        return raw


def fix_nr_kat(val):
    if val is None or pd.isna(val):
        return ''
    v = str(val)
    v = v.replace('*', '')
    v = re.sub(r'\s+', '', v)
    if v.startswith('0'):
        v = 'O' + v[1:]
    if re.fullmatch(r'\d+', v):
        v = 'O' + v
    return v


def fix_wiazka_nr(val):
    if not val:
        return ''
    v = str(val)
    if v.startswith('0'):
        v = 'O' + v[1:]
    return v


def fix_demontaz(row):
    if 'Demontaż zderzaka' in row and row['Demontaż zderzaka'] and row['Demontaż zderzaka'].strip().lower() == 'a':
        row['Demontaż zderzaka'] = 'T'
    if 'Zderzak cięty' in row and row['Zderzak cięty'] and '"' in row['Zderzak cięty']:
        row['Zderzak cięty'] = 'T'
    return row


# ---------------------------------------------------------
# 🔥 GŁÓWNA PĘTLA CZYSZCZENIA
# ---------------------------------------------------------

for df_name, df in [('base', base), ('new', new)]:

    # NIE czyścimy NR KAT. clean_textem
    for col in df.columns:
        if col not in ['MODEL', 'NR KAT.']:
            df[col] = df[col].apply(clean_text)

    # MODEL
    if 'MODEL' in df.columns:
        df['MODEL'] = df['MODEL'].apply(clean_model)

        # 🔥 DODATKOWA POPRAWKA TYLKO DLA HYUNDAI (H100/H200/H300/H350)
        if 'MARKA' in df.columns:
            mask_hyundai = df['MARKA'].str.upper().str.contains('HYUNDAI', na=False)
            df.loc[mask_hyundai, 'MODEL'] = df.loc[mask_hyundai, 'MODEL'].apply(fix_hyundai_model)

    # NR KAT.
    if 'NR KAT.' in df.columns:
        df['NR KAT.'] = df['NR KAT.'].apply(fix_nr_kat)

    # CZAS MONTAŻU
    if 'Czas montażu [h]' in df.columns:
        df['Czas montażu [h]'] = df['Czas montażu [h]'].apply(format_time)

    # ROK PRODUKCJI
    if 'ROK PRODUKCJI' in df.columns:
        df['ROK PRODUKCJI'] = df['ROK PRODUKCJI'].apply(format_rok_produkcji)

    # CENA
    if 'Cena netto [zł]' in df.columns:
        df['Cena netto [zł]'] = df['Cena netto [zł]'].str.replace('zł', '', case=False, regex=False).str.strip()

    # WIĄZKI
    if 'NR KAT. WIĄZKI 7-bieg.' in df.columns:
        df['NR KAT. WIĄZKI 7-bieg.'] = df['NR KAT. WIĄZKI 7-bieg.'].apply(fix_wiazka_nr)

    if 'NR KAT. WIĄZKI 13-bieg.' in df.columns:
        df['NR KAT. WIĄZKI 13-bieg.'] = df['NR KAT. WIĄZKI 13-bieg.'].apply(fix_wiazka_nr)

    # CENY WIĄZEK
    if 'Cena netto [zł].1' in df.columns and 'NR KAT. WIĄZKI 7-bieg.' in df.columns:
        df['Cena netto [zł].1'] = df.apply(
            lambda r: clean_price(r['Cena netto [zł].1'])
            if str(r['NR KAT. WIĄZKI 7-bieg.']).strip() not in ['', 'nan']
            else '',
            axis=1
        )

    if 'Cena netto [zł].2' in df.columns and 'NR KAT. WIĄZKI 13-bieg.' in df.columns:
        df['Cena netto [zł].2'] = df.apply(
            lambda r: clean_price(r['Cena netto [zł].2'])
            if str(r['NR KAT. WIĄZKI 13-bieg.']).strip() not in ['', 'nan']
            else '',
            axis=1
        )

    # HOMOLOGACJA
    if 'NUMER HOMOLOGACJI' in df.columns:
        df['NUMER HOMOLOGACJI'] = df['NUMER HOMOLOGACJI'].apply(format_homologation)

    # 🔥 SPECJALNA KOREKTA TYLKO DLA RODZAJ NADWOZIA
    if 'RODZAJ NADWOZIA' in df.columns:
        df['RODZAJ NADWOZIA'] = df['RODZAJ NADWOZIA'].str.replace(r'\s+,', ',', regex=True)
        df['RODZAJ NADWOZIA'] = df['RODZAJ NADWOZIA'].str.replace(',', ', ', regex=False)
        df['RODZAJ NADWOZIA'] = df['RODZAJ NADWOZIA'].str.replace(r'\s+', ' ', regex=True).str.strip()

    # 🔥 CZYSZCZENIE KOLUMN NUMERYCZNYCH
    numeric_cols = [
        'D [kN]',
        'Masa hol. Przyczepy [kg]',
        'Nacisk na kulę [kg]',
        'Masa haka [kg]'
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_numeric)


# ---------------------------------------------------------
# 🔥 BUDOWANIE KLUCZA
# ---------------------------------------------------------

def normalize_key(val):
    return re.sub(r'\s+', '', str(val).upper()) if val else ''

def safe_col(df, col):
    if col in df.columns:
        return df[col].apply(normalize_key)
    return pd.Series([''] * len(df))

def build_key(df):
    return (
        safe_col(df, 'NR KAT.') + '|' +
        safe_col(df, 'KOD KRESKOWY') + '|' +
        safe_col(df, 'MARKA') + '|' +
        safe_col(df, 'MODEL')
    )

base['KEY'] = build_key(base)
new['KEY'] = build_key(new)


# ---------------------------------------------------------
# 🔥 ŁĄCZENIE
# ---------------------------------------------------------

final_rows = []
base_indexed = base.groupby('KEY').first()

for _, row in new.iterrows():
    key = row['KEY']
    r = row.to_dict()

    if key in base_indexed.index:
        old = base_indexed.loc[key]
        r['NADWOZIE'] = old.get('NADWOZIE', '')
        r['STATUS'] = 'BEZ ZMIAN'
        r['ZMIANY'] = ''
    else:
        r['STATUS'] = 'NOWE'
        r['ZMIANY'] = ''

    r = fix_demontaz(r)
    final_rows.append(r)

final = pd.DataFrame(final_rows)

final.to_csv('HAKI_UPDATED.csv', index=False)

print("✅ GOTOWE")
print(f"Rekordy: {len(final)}")


✅ GOTOWE
Rekordy: 3996
